# Geração de Bases Filtradas — RioMobiAnalytics

Aplica todos os filtros identificados no diagnóstico e salva os arquivos prontos para uso no pipeline.

**Filtros aplicados:**
- Geográfico: apenas paradas dentro de **40 km** do Centro do Rio (Praça XV)
- Serviço: apenas **U_REG** (dias úteis)
- Trips: **1 trip representativa por rota** (grafo final idêntico, ~97% menos dados)
- 1746: apenas reclamações dentro do mesmo raio geográfico

---

## 0. Imports e Configuração

In [ ]:
import pandas as pd
import math
import os
from datetime import datetime

# ---------------------------------------------------------------------------
# Caminhos de entrada (originais)
# ---------------------------------------------------------------------------
BASE_PATH = "/Workspace/Users/vinicius.maciel@ifood.com.br/pessoal/projeto poli"

IN_STOPS       = f"{BASE_PATH}/stops.txt"
IN_ROUTES      = f"{BASE_PATH}/routes.txt"
IN_TRIPS       = f"{BASE_PATH}/trips.txt"
IN_STOP_TIMES  = f"{BASE_PATH}/stop_times.txt"
IN_CALENDAR    = f"{BASE_PATH}/calendar.txt"
IN_1746        = f"{BASE_PATH}/chamados_v2.csv"

# ---------------------------------------------------------------------------
# Caminho de saída (bases filtradas)
# ---------------------------------------------------------------------------
OUTPUT_DIR = f"{BASE_PATH}/filtrado"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_STOPS      = f"{OUTPUT_DIR}/stops.txt"
OUT_ROUTES     = f"{OUTPUT_DIR}/routes.txt"
OUT_TRIPS      = f"{OUTPUT_DIR}/trips.txt"
OUT_STOP_TIMES = f"{OUTPUT_DIR}/stop_times.txt"
OUT_CALENDAR   = f"{OUTPUT_DIR}/calendar.txt"
OUT_1746       = f"{OUTPUT_DIR}/chamados_v2.csv"

# ---------------------------------------------------------------------------
# Parâmetros do recorte
# ---------------------------------------------------------------------------
CENTER_LAT     = -22.9035   # Praça XV — Centro do Rio de Janeiro
CENTER_LON     = -43.1729
RADIUS_KM      = 40.0
SERVICE_ID     = "U_REG"    # apenas dias úteis
EARTH_RADIUS_M = 6_371_000

def haversine_m(lat1, lon1, lat2, lon2):
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1))
         * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    return EARTH_RADIUS_M * 2 * math.asin(math.sqrt(a))

print("Configuração carregada.")
print(f"Entrada  : {BASE_PATH}")
print(f"Saída    : {OUTPUT_DIR}")
print(f"Raio     : {RADIUS_KM} km  |  Serviço: {SERVICE_ID}")

---
## 1. `stops.txt` — Filtro Geográfico (40 km)

In [ ]:
df_stops = pd.read_csv(IN_STOPS)

total_stops = len(df_stops)
print(f"Lido     : {total_stops:,} paradas")

df_stops["_dist_m"] = df_stops.apply(
    lambda r: haversine_m(CENTER_LAT, CENTER_LON, r.stop_lat, r.stop_lon), axis=1
)

df_stops_ok = df_stops[df_stops["_dist_m"] <= RADIUS_KM * 1000].drop(columns=["_dist_m"])

# IDs válidos — usados nos filtros seguintes
valid_stop_ids = set(df_stops_ok["stop_id"].astype(str))

df_stops_ok.to_csv(OUT_STOPS, index=False)

print(f"Filtrado : {len(df_stops_ok):,} paradas ({len(df_stops_ok)/total_stops*100:.1f}%)")
print(f"Removido : {total_stops - len(df_stops_ok):,} paradas fora do raio")
print(f"Salvo em : {OUT_STOPS}")

---
## 2. `calendar.txt` — Apenas U_REG

In [ ]:
df_calendar = pd.read_csv(IN_CALENDAR)

print(f"Lido     : {len(df_calendar)} serviços")
display(df_calendar)

df_calendar_ok = df_calendar[df_calendar["service_id"] == SERVICE_ID]
df_calendar_ok.to_csv(OUT_CALENDAR, index=False)

print(f"\nFiltrado : {len(df_calendar_ok)} serviço(s) mantido(s)")
print(f"Salvo em : {OUT_CALENDAR}")

---
## 3. `trips.txt` — Apenas U_REG + 1 Trip Representativa por Rota

In [ ]:
df_trips = pd.read_csv(IN_TRIPS, dtype={"trip_id": str})

total_trips = len(df_trips)
print(f"Lido     : {total_trips:,} trips")

# Passo 1: filtrar apenas U_REG
df_trips_ureg = df_trips[df_trips["service_id"] == SERVICE_ID].copy()
print(f"U_REG    : {len(df_trips_ureg):,} trips")

# Passo 2: 1 trip representativa por rota
# Pegamos a primeira trip de cada route_id (qualquer uma serve — percurso é o mesmo)
df_trips_ok = df_trips_ureg.groupby("route_id", as_index=False).first()

# IDs das trips selecionadas — usados no filtro do stop_times
selected_trip_ids = set(df_trips_ok["trip_id"])

df_trips_ok.to_csv(OUT_TRIPS, index=False)

print(f"Filtrado : {len(df_trips_ok):,} trips (1/rota)")
print(f"Removido : {total_trips - len(df_trips_ok):,} trips redundantes")
print(f"Redução  : {(1 - len(df_trips_ok)/total_trips)*100:.1f}%")
print(f"Salvo em : {OUT_TRIPS}")

---
## 4. `routes.txt` — Apenas Rotas com Trips Selecionadas

In [ ]:
df_routes = pd.read_csv(IN_ROUTES)

total_routes = len(df_routes)
print(f"Lido     : {total_routes:,} rotas")

# Manter apenas rotas que aparecem nas trips selecionadas
valid_route_ids = set(df_trips_ok["route_id"].astype(str))
df_routes_ok = df_routes[df_routes["route_id"].astype(str).isin(valid_route_ids)]

df_routes_ok.to_csv(OUT_ROUTES, index=False)

print(f"Filtrado : {len(df_routes_ok):,} rotas ({len(df_routes_ok)/total_routes*100:.1f}%)")
print(f"Salvo em : {OUT_ROUTES}")

---
## 5. `stop_times.txt` — Filtro Combinado ⚠️ Arquivo pesado (~75 MB)

In [ ]:
# Leitura em chunks para não estourar memória
# Filtros aplicados em streaming:
#   1. trip_id deve estar em selected_trip_ids (1 trip/rota, U_REG)
#   2. stop_id deve estar em valid_stop_ids (dentro do raio geográfico)

print(f"Lendo {IN_STOP_TIMES} em chunks...")

CHUNK_SIZE = 100_000
chunks_ok = []
total_lido = 0
total_mantido = 0

for chunk in pd.read_csv(
    IN_STOP_TIMES,
    dtype={"stop_id": str, "trip_id": str},
    chunksize=CHUNK_SIZE
):
    total_lido += len(chunk)

    chunk_ok = chunk[
        chunk["trip_id"].isin(selected_trip_ids) &
        chunk["stop_id"].isin(valid_stop_ids)
    ]

    total_mantido += len(chunk_ok)
    if len(chunk_ok) > 0:
        chunks_ok.append(chunk_ok)

    print(f"  Lido: {total_lido:,}  |  Mantido até agora: {total_mantido:,}", end="\r")

print(f"\nTotal lido    : {total_lido:,}")
print(f"Total mantido : {total_mantido:,} ({total_mantido/total_lido*100:.1f}%)")
print(f"Removido      : {total_lido - total_mantido:,} ({(1-total_mantido/total_lido)*100:.1f}%)")

In [ ]:
# Concatenar e salvar
df_stop_times_ok = pd.concat(chunks_ok, ignore_index=True)

df_stop_times_ok.to_csv(OUT_STOP_TIMES, index=False)

print(f"Linhas salvas : {len(df_stop_times_ok):,}")
print(f"Salvo em      : {OUT_STOP_TIMES}")
df_stop_times_ok.head(3)

---
## 6. `chamados_v2.csv` — Filtro Geográfico (40 km)

In [ ]:
df_1746 = pd.read_csv(IN_1746)

total_1746 = len(df_1746)
print(f"Lido     : {total_1746:,} reclamações")

# Filtrar por coordenadas válidas + raio geográfico
df_1746_valid = df_1746.dropna(subset=["latitude", "longitude"]).copy()

df_1746_valid["_dist_m"] = df_1746_valid.apply(
    lambda r: haversine_m(CENTER_LAT, CENTER_LON, r.latitude, r.longitude), axis=1
)

df_1746_ok = df_1746_valid[
    df_1746_valid["_dist_m"] <= RADIUS_KM * 1000
].drop(columns=["_dist_m"])

df_1746_ok.to_csv(OUT_1746, index=False)

print(f"Filtrado : {len(df_1746_ok):,} reclamações ({len(df_1746_ok)/total_1746*100:.1f}%)")
print(f"Removido : {total_1746 - len(df_1746_ok):,} (sem coordenadas ou fora do raio)")
print(f"Salvo em : {OUT_1746}")